<a href="https://colab.research.google.com/github/OskarDroescherWarwick/SQLite_versus_TinyDB/blob/main/SQLite_versus_TinyDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.01 Schema on write or schema on read?

You will load the same data into two small databases and ask the same questions of each:

| Store | Kind | Python library |
| --- | --- | --- |
| SQLite | relational (tables, SQL) | `sqlite3` (built in) |
| TinyDB | document store (JSON-like documents - NoSQL) | `tinydb` |


The point is **not speed**. It is to see two things:
1. **Schema on write versus schema on read.** *Schema on write* means the structure of the data is declared up front and the database **enforces it when data is stored**. This is the approach taken in relational databases. *Schema on read* means data is stored as it arrives and **structure is applied by whoever reads it**. This is the approach taken in most NoSQL approaches.
2. **Where the work goes.** In SQL a join is one clause and the database does it. In TinyDB there are no joins, so **you write the join yourself in Python.**

**By the end you should be able to:**
- Say which store enforced the rules, and what happened when bad data arrived.
- Translate a plain-English question into SQL and into Python code for a document store.
- Explain what the database did for you in SQLite that you had to write yourself in TinyDB.
- Give a situation where schema on read is the better choice.

**Please note**
- SQLite is **loosely typed by default**: an ordinary table will happily store the text `"12 mins"` in a number column. It enforces types only in tables created with the `STRICT` keyword, which needs a recent SQLite version (as far as I know 3.37 or later; the notebook checks your version and tells you).
- TinyDB's own documentation describes it as a "document oriented" database ([tinydb.readthedocs.io](https://tinydb.readthedocs.io/en/latest/)). It is a small library for learning and light use, so what you see here says little about large document databases. Worth noting that the difference between key value databases and document orientated databases is fairly insignificant these days.

## 1. Set up
This creates a small **synthetic bike-hire** dataset: 12 stations, 60 bikes and 2,000 trips. Eight of the bikes were never used, which matters for one of the questions.

In [1]:
%pip install -q tinydb

In [10]:
import sqlite3
import numpy as np
import pandas as pd
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

rng = np.random.default_rng(42) # set a fixed seed for random number generation. Means results are replicable.

# Dataframe of station data
stations = pd.DataFrame({
    "station_id":   [f"S{i:02d}" for i in range(1, 13)],                                                  # 12 stations with format S0[i] e.g. S01, S02
    "station_name": ["Market Square", "Railway Station", "Canal Basin", "University Gate", "Park Road",   # 12 station names station[i]=station[1] relates to first station name
                     "Library Corner", "Hospital Entrance", "Riverside Walk", "Old Town Hall",
                     "Retail Park", "Sports Centre", "Cathedral Close"],
    "zone":         ["Centre", "Centre", "North", "West", "South", "Centre",                              # zone in which each station lies
                     "East", "South", "Centre", "North", "West", "Centre"],
    "capacity":     rng.integers(10, 31, 12),                                                             # capacity = number of bikes that can be stored in each station
})

#print("Station IDs:")
#print(stations.station_id)
#print("Station names:")
#print(stations.station_name)
#print("Station zones:")
#print(stations.zone)
#print("Station capacities:")
#print(stations.capacity)


# Dataframe of bike data
bikes = pd.DataFrame({
    "bike_id":       [f"B{i:03d}" for i in range(1, 61)],                        # bike ID with format B0[i] e.g. B023, B019
    "bike_type":     rng.choice(["Standard", "Electric"], 60, p=[0.7, 0.3]),     # random allocate each bike a bike type (Standard or Electric) => 70% of bikes shall be Standard and 30% Electric
    "purchase_year": rng.integers(2021, 2025, 60),
})


N_TRIPS = 2000                                                                   # number of trips
used_bikes = bikes["bike_id"].iloc[:52]                                          # used_bikes = first 52 bikes, the last 8 bikes are never used


# Dataframe of individual trips
trips = pd.DataFrame({
    "trip_id":          np.arange(1, N_TRIPS + 1),                                 # trip ID => start at 1 for the number of trips (2000) count +1 for each trip id, so that trip ids => 1, 2, 3, ...
    "bike_id":          rng.choice(used_bikes, N_TRIPS),                           # assign trip a random bike_id from the bikes dataframe but only from the used_bikes ids
    "start_station_id": rng.choice(stations["station_id"], N_TRIPS),               # assign trip a random start by using station_id from stations dataframe
    "end_station_id":   rng.choice(stations["station_id"], N_TRIPS),               # assign trips a random end by using station_id stom stations dataframe
    "user_type":        rng.choice(["Member", "Casual"], N_TRIPS, p=[0.65, 0.35]), # random allocate each trip a user_type (Member or Casual) => 65% of trips by Member and 35% by Casual riders
    "start_hour":       rng.integers(0, 24, N_TRIPS),                              # assign a start time as integer between 0 and 24
    "duration_min":     rng.gamma(2.0, 8.0, N_TRIPS).round(1),                     # assign duration of trip in min between 2-8min and round to 1 decimal point
})

print(len(stations), "stations,", len(bikes), "bikes,", len(trips), "trips")       # print number of stations, bikes, and trips
trips.head(10)                                                                     # print the first ten rows of the trips dataframe


12 stations, 60 bikes, 2000 trips


,trip_id,bike_id,start_station_id,end_station_id,user_type,start_hour,duration_min
0,1,B032,S04,S09,Member,0,7.9
1,2,B019,S03,S01,Member,16,11.3
2,3,B050,S08,S07,Member,21,6.0
3,4,B005,S10,S06,Member,15,3.3
4,5,B018,S04,S07,Casual,14,9.5
5,6,B007,S11,S02,Member,19,37.5
6,7,B018,S07,S12,Casual,23,44.3
7,8,B051,S03,S03,Casual,20,6.7
8,9,B020,S05,S09,Member,5,14.6
9,10,B048,S01,S09,Casual,13,18.4


## 2. Same data, two stores
**Before running anything, predict.** Below are five **bad rows**. For each, will each store **accept** or **reject** it when asked to store it? Write `"accept"` or `"reject"` over the underlined string below (i.e. "____" might become "accept"). Not marked; the aim is to find where your intuition is wrong.

In [20]:
predictions = {
    "missing duration":           {"SQLite": "reject", "TinyDB": "accept"},
    "duration is text":           {"SQLite": "reject", "TinyDB": "accept"},
    "negative duration":          {"SQLite": "reject", "TinyDB": "accept"},
    "station that doesn't exist": {"SQLite": "reject", "TinyDB": "accept"},
    "extra field not in schema":  {"SQLite": "reject", "TinyDB": "accept"},
}

In [14]:
GOOD = {"trip_id": 9001, "bike_id": "B001", "start_station_id": "S01", "end_station_id": "S02",
        "user_type": "Member", "start_hour": 9, "duration_min": 12.0}

# make bad variants
def variant(**changes):
    doc = dict(GOOD); doc.update(changes); return doc

bad_docs = {
    "missing duration":          {k: v for k, v in GOOD.items() if k != "duration_min"} | {"trip_id": 9001},
    "duration is text":          variant(trip_id=9002, duration_min="12 mins"),
    "negative duration":         variant(trip_id=9003, duration_min=-5.0),
    "station that doesn't exist": variant(trip_id=9004, start_station_id="S99"),
    "extra field not in schema": variant(trip_id=9005, notes="bike returned damaged"),
}

### 2a. SQLite: schema on write
Read the `CREATE TABLE` statements. The rules live **in the database**: types, `NOT NULL`, allowed values (`CHECK`), and `REFERENCES` (foreign keys, meaning "this must match a row in another table").

In [15]:
print("SQLite version:", sqlite3.sqlite_version)
STRICT = " STRICT" if sqlite3.sqlite_version_info >= (3, 37, 0) else ""
if not STRICT:
    print("This SQLite is older than 3.37, so column types are NOT enforced. Expect the 'duration is text' row to be accepted.")

sql_con = sqlite3.connect(":memory:")
sql_con.execute("PRAGMA foreign_keys = ON")                                       # SQLite only checks foreign keys if you switch this on
sql_con.executescript(f"""
CREATE TABLE stations (
    station_id   TEXT PRIMARY KEY,
    station_name TEXT NOT NULL,
    zone         TEXT NOT NULL,
    capacity     INTEGER NOT NULL
){STRICT};

CREATE TABLE bikes (
    bike_id       TEXT PRIMARY KEY,
    bike_type     TEXT NOT NULL CHECK (bike_type IN ('Standard', 'Electric')),
    purchase_year INTEGER NOT NULL
){STRICT};

CREATE TABLE trips (
    trip_id          INTEGER PRIMARY KEY,
    bike_id          TEXT NOT NULL REFERENCES bikes(bike_id),
    start_station_id TEXT NOT NULL REFERENCES stations(station_id),
    end_station_id   TEXT NOT NULL REFERENCES stations(station_id),
    user_type        TEXT NOT NULL CHECK (user_type IN ('Member', 'Casual')),
    start_hour       INTEGER NOT NULL CHECK (start_hour BETWEEN 0 AND 23),
    duration_min     REAL NOT NULL CHECK (duration_min > 0)
){STRICT};
""")

for name, frame in [("stations", stations), ("bikes", bikes), ("trips", trips)]:
    marks = ",".join("?" * frame.shape[1])
    sql_con.executemany(f"INSERT INTO {name} VALUES ({marks})", list(frame.itertuples(index=False, name=None)))
sql_con.commit()
print({t: sql_con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ["stations", "bikes", "trips"]})

SQLite version: 3.45.1
{'stations': 12, 'bikes': 60, 'trips': 2000}


Now try to store the five bad rows in SQLite. Any row that gets in is removed again so the data stays clean.

In [16]:
def try_insert_sqlite(doc):
    cols = list(doc)
    sql = f"INSERT INTO trips ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})"
    try:
        sql_con.execute(sql, [doc[c] for c in cols])
        sql_con.execute("DELETE FROM trips WHERE trip_id = ?", (doc["trip_id"],))   # keep the data clean
        sql_con.commit()
        return "accept", ""
    except sqlite3.Error as e:
        return "reject", str(e)

sqlite_outcomes = {}
for name, doc in bad_docs.items():
    result, msg = try_insert_sqlite(doc)
    sqlite_outcomes[name] = result
    print(f"{name:28s} SQLite {result.upper():7s} {msg}")

missing duration             SQLite REJECT  NOT NULL constraint failed: trips.duration_min
duration is text             SQLite REJECT  cannot store TEXT value in REAL column trips.duration_min
negative duration            SQLite REJECT  CHECK constraint failed: duration_min > 0
station that doesn't exist   SQLite REJECT  FOREIGN KEY constraint failed
extra field not in schema    SQLite REJECT  table trips has no column named notes


### 2b. TinyDB: schema on read
TinyDB has no `CREATE TABLE`. We just hand it dictionaries. It does have named tables (`db.table("trips")`) so we keep the same three groups of data. We use in-memory storage so there is no file.

We use a **second copy** (`scratch`) for the bad rows so the main data stays clean.

In [17]:
def build_tiny():
    db = TinyDB(storage=MemoryStorage)
    db.table("stations").insert_multiple(stations.to_dict("records"))
    db.table("bikes").insert_multiple(bikes.to_dict("records"))
    db.table("trips").insert_multiple(trips.to_dict("records"))
    return db

tiny_db = build_tiny()
print({t: len(tiny_db.table(t)) for t in ["stations", "bikes", "trips"]})
print(tiny_db.table("stations").all()[0])

{'stations': 12, 'bikes': 60, 'trips': 2000}
{'station_id': 'S01', 'station_name': 'Market Square', 'zone': 'Centre', 'capacity': 11}


In [18]:
scratch = build_tiny()
tiny_outcomes = {}
for name, doc in bad_docs.items():
    scratch.table("trips").insert(doc)
    tiny_outcomes[name] = "accept"
    print(f"{name:28s} TinyDB accepted it")
print("trips in scratch:", len(scratch.table("trips")), "(was 2000)")

missing duration             TinyDB accepted it
duration is text             TinyDB accepted it
negative duration            TinyDB accepted it
station that doesn't exist   TinyDB accepted it
extra field not in schema    TinyDB accepted it
trips in scratch: 2005 (was 2000)


In [21]:
check = pd.DataFrame({
    "you predicted: SQLite": {k: v["SQLite"] for k, v in predictions.items()},
    "SQLite did":            sqlite_outcomes,
    "you predicted: TinyDB": {k: v["TinyDB"] for k, v in predictions.items()},
    "TinyDB did":            tiny_outcomes,
})
check

,you predicted: SQLite,SQLite did,you predicted: TinyDB,TinyDB did
missing duration,reject,reject,accept,accept
duration is text,reject,reject,accept,accept
negative duration,reject,reject,accept,accept
station that doesn't exist,reject,reject,accept,accept
extra field not in schema,reject,reject,accept,accept


**The accepted rows are not harmless.** TinyDB accepted them, but any code that reads them must cope. Here is a simple "average duration by user type" written for TinyDB. Run it on the clean data and then on the scratch copy:

In [22]:
def naive_average_by_user_type(db):
    total, count = {}, {}
    for t in db.table("trips"):
        u = t["user_type"]
        total[u] = total.get(u, 0) + t["duration_min"]
        count[u] = count.get(u, 0) + 1
    return {u: round(total[u] / count[u], 3) for u in total}

print("on the clean data:", naive_average_by_user_type(tiny_db))
try:
    print(naive_average_by_user_type(scratch))
except Exception as e:
    print("on the scratch data it failed:", type(e).__name__, e)

on the clean data: {'Member': 16.04, 'Casual': 15.915}
on the scratch data it failed: KeyError 'duration_min'


Nothing was wrong with the *code* when the data was clean. The schema was never written down anywhere, so the first bad row broke it, and each reader of this data has to defend against such rows separately.

**Short answer cell:** In one or two sentences, where does the schema live in SQLite, and where does it live in TinyDB?

*Your answer:* In SQLite the schema lives in the DB itself, whereas in TinyDB the schema lives in the application level e.g. Python code

## 3. The same question, in English, SQL and Python
Now the main exercise. Each question is stated **in plain English first**. Your job is to translate it twice: into **SQL** for SQLite, and into **Python** for TinyDB. Both stores hold the same clean data, so the answers must match. A check cell after each question tells you.

Tips for TinyDB:
- `db.table("trips")` gives a table you can loop over (`for t in table:`) or search (`table.search(Query().user_type == "Casual")`, or `table.count(...)`).
- There is **no join and no group-by**. Use dictionaries for lookups and running totals.
- Return results in the exact shape stated so the check can compare them.

Helper used by the checks:

In [23]:
def compare(label, from_sqlite, from_tiny):
    same = from_sqlite == from_tiny
    print(f"{label}: {'the two stores AGREE' if same else 'the two stores DIFFER'}")
    print("  SQLite:", from_sqlite)
    print("  TinyDB:", from_tiny)

### Q1. *"How many trips were made by Casual users?"* (worked example, run as is)
One table, so it is almost the same in both. Returns a number.

In [24]:
def q1_sqlite():
    return sql_con.execute("SELECT COUNT(*) FROM trips WHERE user_type = 'Casual'").fetchone()[0]

def q1_tiny(db=tiny_db):
    return db.table("trips").count(Query().user_type == "Casual")

compare("Q1", q1_sqlite(), q1_tiny())

Q1: the two stores AGREE
  SQLite: 663
  TinyDB: 663


### Q2. *"What is the average trip duration, in minutes, for trips starting in each zone?"*
Needs **two tables**: the trips (which have a start station id) and the stations (which know the zone). Returns a dictionary `{zone: average rounded to 3 decimals}`.

The SQLite version is given. **You write the TinyDB version.** Hints: build a dictionary `{station_id: zone}` from the stations table, loop over the trips, and keep a running total and count for each zone.

In [41]:
def q2_sqlite():
    rows = sql_con.execute("""
        SELECT s.zone, AVG(t.duration_min)
        FROM trips t
        JOIN stations s ON t.start_station_id = s.station_id
        GROUP BY s.zone""").fetchall()
    return {zone: round(avg, 3) for zone, avg in rows}

In [77]:
def q2_tiny(db=tiny_db):
  # TODO: look up each trip's zone, then average the duration per zone

  # dictionary to map station_id of each station to its zone, so we don´t have to look in DB every time
  station_to_zone = {s['station_id']: s['zone'] for s in db.table("stations").all()}

  # create two empty dictionaries
  # zone_total = total of minutes spent riding in each zone
  # zone_count = how many trips happened in each zone
  zone_total, zone_count = {}, {}


  for trip in db.table("trips"):                                                         # iterate over each single trip (bike ride)
      start_id = trip['start_station_id']                                                # get the start station id of our current trip
      zone = station_to_zone[start_id]                                                   # get the zone name that correlates with the start station id we just retrieved
      duration = trip['duration_min']                                                    # get the length (duration) of our current trip
      zone_total[zone] = zone_total.get(zone, 0) + duration                              # add the duration we retrieved in the step before to the zone_total for our specific zone of that trip
      zone_count[zone] = zone_count.get(zone, 0) + 1                                     # add +1 to the zone_count to represent that one more bike ride happened in that specific zone => to calc avg later an avg = total_duration / total_count

  return {zone: round(zone_total[zone] / zone_count[zone], 3) for zone in zone_total}    # after we iterated over all trips => for each zone calc avg = total min / total trips => round to 3 decimal points

compare("Q2", q2_sqlite(), q2_tiny())

Q2: the two stores AGREE
  SQLite: {'Centre': 15.971, 'East': 16.289, 'North': 14.648, 'South': 16.393, 'West': 17.191}
  TinyDB: {'West': 17.191, 'North': 14.648, 'South': 16.393, 'East': 16.289, 'Centre': 15.971}


### Q3. *"How many trips by Electric bikes ended in each zone?"*
Needs **three tables**: trips, bikes (for the bike type) and stations (for the zone of the **end** station). Returns `{zone: number_of_trips}`.

You write **both** versions.

In [116]:
def q3_sqlite():
    # TODO: join trips to bikes and to stations (using end_station_id), filter, group, count
    rows = sql_con.execute("""
        SELECT s.zone, COUNT(t.trip_id)
        FROM trips t
        JOIN stations s ON t.end_station_id = s.station_id
        JOIN bikes b ON t.bike_id = b.bike_id
        WHERE b.bike_type = 'Electric'
        GROUP BY s.zone""").fetchall()
    return {zone: count for zone, count in rows}



def q3_tiny(db=tiny_db):
    # TODO: build two lookup dictionaries (bike -> type, station -> zone), then loop over trips

    # dictionary to map bike_id of each bike to its type
    bike_to_type = {b['bike_id']: b['bike_type'] for b in db.table("bikes").all()}

    # dictionary to map station_id of each station to its zone
    station_to_zone = {s['station_id']: s['zone'] for s in db.table("stations").all()}

    # keep track of Electric bike rides in each zone
    zone_total = {}

    for trip in db.table("trips"):                                                   # for each trip
      end_id = trip['end_station_id']                                                # get the end station id of our current trip
      zone = station_to_zone[end_id]                                                 # get the zone name that correlates to the end station id of our current trip
      bike_id = trip['bike_id']                                                      # get the bike id of our bike for that current trip
      bike_type = bike_to_type[bike_id]                                              # get the bike type of out bike
      if bike_type == "Electric":                                                    # if the bike type is Electric, then
        zone_total[zone] = zone_total.get(zone, 0) + 1                               # add 1 to the total counter of electric rides in that zone
      else:                                                                          # else
        continue                                                                     # do nothing and continue with the next trip
    return {zone: zone_total[zone] for zone in zone_total}                           # return the total number of electric rides for each zone


compare("Q3", q3_sqlite(), q3_tiny())

Q3: the two stores AGREE
  SQLite: {'Centre': 276, 'East': 62, 'North': 112, 'South': 125, 'West': 134}
  TinyDB: {'Centre': 276, 'East': 62, 'South': 125, 'West': 134, 'North': 112}


### Q4. *"Which bikes (id and type) have never been used for a trip?"*
Something that is **not in** another table. Returns a list of `(bike_id, bike_type)` tuples sorted by bike id.

SQL hint: a `LEFT JOIN` from bikes to trips keeps bikes with no matching trip; those rows have `NULL` in the trip columns. TinyDB hint: collect the set of bike ids that appear in trips, then keep the bikes not in that set.

In [190]:
def q4_sqlite():
    rows = sql_con.execute("""
        SELECT b.bike_id, b.bike_type
        FROM bikes b
        LEFT JOIN trips t ON b.bike_id = t.bike_id
        WHERE t.bike_id IS NULL
        ORDER BY b.bike_id""").fetchall()
    return [(bike_id, bike_type) for bike_id, bike_type in rows]                    # Return a list of tuples instead of a dictionary


def q4_tiny(db=tiny_db):
  # TODO
  # dictionary to map bike_id of each bike to its type
  bike_to_type = {b['bike_id']: b['bike_type'] for b in db.table("bikes").all()}

  # create two lists
  bikes_used = []                                                                  # use list[]
  bikes_not_used = []                                                              # use list[] so we can add tuples in the end

  for trip in db.table("trips"):                                                   # for each trip
    bike_id = trip['bike_id']                                                      # get the bike id of our bike for that current trip
    bikes_used.append(bike_id)                                                     # add the bike id to our list bikes_used

  for bike in db.table("bikes"):                                                   # for each bike
    bike_id = bike['bike_id']                                                      # get the bike id of our current bike
    bike_type = bike['bike_type']                                                  # get the bike type of our current bike
    if bike_id in bikes_used:                                                      # if current bike id is part of set() bikes_used, then
      continue                                                                     # do nothing and continue with the next bike
    else:                                                                          # else
      bikes_not_used.append((bike_id, bike_type))                                  # Add the bike id and the bike type as a tuple() to the list bikes_not_used
  return bikes_not_used                                                            # return the dictionary that contains all bike ids and types of bikes that have never been used


compare("Q4", q4_sqlite(), q4_tiny())

Q4: the two stores AGREE
  SQLite: [('B053', 'Standard'), ('B054', 'Standard'), ('B055', 'Standard'), ('B056', 'Standard'), ('B057', 'Standard'), ('B058', 'Electric'), ('B059', 'Standard'), ('B060', 'Standard')]
  TinyDB: [('B053', 'Standard'), ('B054', 'Standard'), ('B055', 'Standard'), ('B056', 'Standard'), ('B057', 'Standard'), ('B058', 'Electric'), ('B059', 'Standard'), ('B060', 'Standard')]


## 4. Reading defensively: schema on read in practice
In Part 2 the naive TinyDB code crashed on the scratch data. SQLite never had to deal with those rows, because it refused them on the way in.

Write a safe version of Q2 for TinyDB that applies, at read time, the rules SQLite applied at write time. It should skip any trip where:
- `duration_min` is missing, or
- `duration_min` is not a number (`int` or `float`), or
- `duration_min` is zero or negative, or
- the start station does not exist.

Return two things: the same dictionary as Q2, and a list of `(trip_id, reason)` for each skipped trip.

In [192]:
def q2_tiny_safe(db):
  # TODO: like q2_tiny, but check each trip against the four rules above and record why it was skipped

  # map stations to zones
  station_to_zone = {s['station_id']: s['zone'] for s in db.table("stations").all()}

  # add counter dictionaries and list for skipped trips
  zone_total = {}                                                                # zone_total is sum of duration in each zone
  zone_count = {}                                                                # zone_count is sum of trips in each zone
  skipped = []                                                                   # list where we store information if a trip was skipped and why

  # iterate over each trip:
  for trip in db.table("trips").all():
      trip_id = trip.get('trip_id')                                              # get the current trip id

      # check if: duration_min is missing
      if 'duration_min' not in trip:
          skipped.append((trip_id, "duration_min is missing"))                   # if true then add trip id and reason to list skipped[] as tuple()
          continue                                                               # else do nothing and continue with next step

      # duration needed for next two rules to check
      duration = trip['duration_min']                                            # get duration of the current trip

      # check if: duration_min is not a number
      if type(duration) not in (int, float):                                     # use type() to strictly check only for int and float values and also reject booleans
          skipped.append((trip_id, "duration_min is not a number"))              # if true, then add trip id and reason to list skipped[] as tuple()
          continue                                                               # else do nothing and continue with next step

      # check if: duration_min is zero or negative
      if duration <= 0:
          skipped.append((trip_id, "duration_min is zero or negative"))          # if true, then add trip id and reason to list skipped[] as tuple()
          continue                                                               # else do nothing and continue with next step

      # check if: the start station does not exist
      start_id = trip.get('start_station_id')                                    # get the start station id (name) of the current trip
      if start_id not in station_to_zone:                                        # if current start station doesn´t match allowed start station from dictionary
          skipped.append((trip_id, "start station does not exist"))              # if true, then add trip id and reason to list skipped[] as tuple()
          continue                                                               # else do nothing and continue with next step

      # if the trip passes all 4 checks, safely process it                       # if current trip is legitimate and passes all 4 checks
      zone = station_to_zone[start_id]                                           # get zone = current trips start id (start station name)
      zone_total[zone] = zone_total.get(zone, 0) + duration                      # add current trips duration to the total duration time for that zone
      zone_count[zone] = zone_count.get(zone, 0) + 1                             # add +1 to counter of how many trips started in that start station name (zone)

  # Calculate averages only for the safe data
  result = {zone: round(zone_total[zone] / zone_count[zone], 3) for zone in zone_total}  # for each zone => print zone name and calc avg trip duration time for that zone

  return result, skipped                                                         # return dictionary result{} and list skipped[] with its tuples

result, skipped = q2_tiny_safe(scratch)
print("result :", result)
print("skipped:", skipped)

result : {'West': 17.191, 'North': 14.648, 'South': 16.393, 'East': 16.289, 'Centre': 15.966}
skipped: [(9001, 'duration_min is missing'), (9002, 'duration_min is not a number'), (9003, 'duration_min is zero or negative'), (9004, 'start station does not exist')]


Four rows were skipped, and the fifth bad row (the one with the extra `notes` field) was **kept**, because its duration and station were fine. SQLite refused that row; TinyDB accepted it and your code counted it, so a zone's average may differ slightly.

Was that the right outcome? There is no technical answer. It is an

![](https://drive.google.com/uc?export=view&id=1JjZejnp5SGPBXChOr2vHWmVQJDWq7HeQ)

(i.e. a business decision) about what counts as a valid trip.

In SQLite someone made that decision once and wrote it into the schema. In TinyDB every reader has to make it again.

## 5. Your own question
Write a question of your own, **in plain English**, that needs **at least one join**. Then translate it into SQL and into Python and check the two agree. Some ideas:
- Which station is the most common start point for Casual users in the Centre zone?
- What is the average trip duration for Electric versus Standard bikes?
- Which zone has the most trips that start and end in the same zone?

In [200]:
my_question = "Which station is the most common start point for Casual users in the Centre zone?"

def mine_sqlite():
  # TODO
  row = sql_con.execute("""
      SELECT t.start_station_id                               -- only return the station id
      FROM trips t                                            -- start with all trips
      JOIN stations s ON t.start_station_id = s.station_id    -- get the station info for each trip
      WHERE t.user_type = 'Casual' AND s.zone = 'Centre'      -- apply filters user_type=Casual and zone=Centre
      GROUP BY t.start_station_id                             -- counter for each trip that starts at that station
      ORDER BY COUNT(*) DESC                                  -- sort by highest count
      LIMIT 1 """).fetchone()                                  # keep only the #1 winner => analog to max()
  return row[0]                                                # fetch that single winning row from the database


def mine_tiny(db=tiny_db):
    # TODO
    # create dictionary to map station_ids to Centre zone
    centre_stations = {s['station_id'] for s in db.table("stations").all() if s['zone'] == 'Centre'}

    # counter for how many times each station is used by a Casual user
    station_counts = {}

    for trip in db.table("trips").all():                                         # for each trip
        if trip.get('user_type') == 'Casual':                                    # check if the user is a casual rider
            start_id = trip['start_station_id']                                  # if true, then get the start station id of current trip
            if start_id in centre_stations:                                      # check if that start station id is in Centre zone dictionary
                station_counts[start_id] = station_counts.get(start_id, 0) + 1   # if true, then add +1 to counter of how many casual rider start at the current station

    # find the station id with the highest count
    # max() loops through the dictionary station_counts, and key=station_counts.get tells it to judge the max based on the values in station_counts
    most_common_station = max(station_counts, key=station_counts.get)

    return most_common_station

compare(my_question, mine_sqlite(), mine_tiny())

Which station is the most common start point for Casual users in the Centre zone?: the two stores AGREE
  SQLite: S12
  TinyDB: S12


## 6. Reflection
1. In SQLite, what did the *database* do for you in Part 2? What did you have to write *yourself* in Part 4 to get similar protection in TinyDB?
2. Which of Q1 to Q4 was the most different to write in the two systems, and why? What did you have to decide in TinyDB that SQL decided for you?
3. Suppose a second analyst writes their own TinyDB code for Q2 and forgets to skip the bad rows. What happens to the team's numbers?
4. Give two situations where schema on read is the better choice. How does this connect to a data lake or the bronze layer (week 2)?
5. What is the cost of schema on write? What would you need to do in SQLite to accept the extra `notes` field legitimately?